<h1 style="color: #5439a7ff; text-align: center;">
  Practica 3: Sensado y análisis inercial
</h1>
<h5 style="text-align: center;">
    Nombre del alumno: Jose Francisco Juarez Aceves 
    <br>
    Materia: Ciencia de Datos para Sensores Inteligentes
</h5>

<h2 style="color: #5439a7ff; text-align: center;">
  Librerias utilizadas
</h3>

In [123]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.spatial.transform import Rotation as R
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier


<h2 style="color: #5439a7ff; text-align: center;">
  Procesamiento y Extraccion de caracteristicas
</h3>

In [79]:
ruta_train = "/home/josejuarez/Documents/Dataset_Inercial/train/*.csv"
csv_train = glob.glob(ruta_train)
ruta_test = "/home/josejuarez/Documents/Dataset_Inercial/test/*.csv"
csv_test = glob.glob(ruta_test)

In [44]:
def filtro_pasabaja(signal, fs=100, cutoff=8, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, signal)

In [45]:
def rename_variables(data):
    data.rename(columns={'GyroX (deg/s)':'GyroX'}, inplace=True)
    data.rename(columns={'GyroY (deg/s)':'GyroY'}, inplace=True)
    data.rename(columns={'GyroZ (deg/s)':'GyroZ'}, inplace=True)
    data.rename(columns={'QuatW':'QuatW'}, inplace=True)
    data.rename(columns={'QuatX':'QuatX'}, inplace=True)
    data.rename(columns={'QuatY':'QuatY'}, inplace=True)
    data.rename(columns={'QuatZ':'QuatZ'}, inplace=True)
    data.rename(columns={'LinAccX (g)':'LinAccX'}, inplace=True)
    data.rename(columns={'LinAccY (g)':'LinAccY'}, inplace=True)
    data.rename(columns={'LinAccZ (g)':'LinAccZ'}, inplace=True)
    data.rename(columns={'TimeStamp (s)':'TimeStamp'}, inplace=True)
    return data


In [46]:
def calculate_mag(data):
    data["LinAccMag"] = np.sqrt(
        data["LinAccX"]**2 +
        data["LinAccY"]**2 +
        data["LinAccZ"]**2
    )

    data["GyroMag"] = np.sqrt(
        data["GyroX"]**2 +
        data["GyroY"]**2 +
        data["GyroZ"]**2
    )

    return data

In [47]:
def max_relative_rotation(seg_mano, seg_antebrazo):
    seg_mano = seg_mano.sort_values("TimeStamp")
    seg_antebrazo = seg_antebrazo.sort_values("TimeStamp")
    
    angles = []
    
    for i in range(min(len(seg_mano), len(seg_antebrazo))):
        q_m = np.array(seg_mano.iloc[i][["QuatX","QuatY","QuatZ","QuatW"]], dtype=float)
        q_a = np.array(seg_antebrazo.iloc[i][["QuatX","QuatY","QuatZ","QuatW"]], dtype=float)
        r_m = R.from_quat(q_m)
        r_a = R.from_quat(q_a)
        r_rel = r_m * r_a.inv()
        angles.append(r_rel.magnitude())
    
    return np.max(angles)


In [48]:
def total_rotation(seg):
    seg = seg.sort_values("TimeStamp")
    
    if len(seg) < 2:
        return np.nan
    
    q_start = np.array(seg.iloc[0][["QuatX","QuatY","QuatZ","QuatW"]], dtype=float)
    q_end   = np.array(seg.iloc[-1][["QuatX","QuatY","QuatZ","QuatW"]], dtype=float)
    r_start = R.from_quat(q_start)
    r_end   = R.from_quat(q_end)
    r_rel = r_end * r_start.inv()
    return r_rel.magnitude()


In [49]:
def time_to_peak_rotation(seg):
    seg = seg.sort_values("TimeStamp")
    q_start = np.array(seg.iloc[0][["QuatX","QuatY","QuatZ","QuatW"]], dtype=float)
    r_start = R.from_quat(q_start)
    
    angles = []
    
    for _, row in seg.iterrows():
        q = np.array(row[["QuatX","QuatY","QuatZ","QuatW"]], dtype=float)
        r = R.from_quat(q)
        r_rel = r * r_start.inv()
        angles.append(r_rel.magnitude())
    angles = np.array(angles)
    idx_peak = np.argmax(angles)
    return seg.iloc[idx_peak]["TimeStamp"]


In [50]:
def rotational_energy(seg):
    seg = seg.sort_values("TimeStamp")
    
    if len(seg) < 2:
        return np.nan

    q_start = np.array(seg.iloc[0][["QuatX","QuatY","QuatZ","QuatW"]],dtype=float)
    r_start = R.from_quat(q_start)
    angles = []
    times = seg["TimeStamp"].to_numpy().copy()
    
    for _, row in seg.iterrows():
        q = np.array(row[["QuatX","QuatY","QuatZ","QuatW"]],dtype=float)
        r = R.from_quat(q)
        r_rel = r * r_start.inv()
        
        angles.append(r_rel.magnitude())
    
    angles = np.array(angles)
    
    return np.trapz(angles**2, times)



In [51]:
def stability_last300ms(seg, t_release):
    seg = seg.sort_values("TimeStamp")
    seg_last = seg[seg["TimeStamp"] >= (t_release)]
    seg_last = seg_last[seg_last["TimeStamp"] <= (t_release + 0.3)]
    if len(seg_last) < 2:
        return np.nan
    
    q_release = np.array(seg_last.iloc[0][["QuatX","QuatY","QuatZ","QuatW"]], dtype=float)
    r_release = R.from_quat(q_release)
    
    angles = []
    
    for _, row in seg_last.iterrows():
        q = np.array(row[["QuatX","QuatY","QuatZ","QuatW"]], dtype=float)
        r = R.from_quat(q)
        r_rel = r * r_release.inv()
        angles.append(r_rel.magnitude())
    
    return np.std(angles)


In [129]:
def extract_features(lista_archivos):

    all_features = []

    for archivo in lista_archivos:
    
        print("\nProcesando:", archivo)
    
        data = pd.read_csv(archivo, encoding="utf-8-sig")
        data.columns = data.columns.str.strip()
        sensor_map = {2: "Mano", 4: "Bicep", 5: "Antebrazo"}
        data["Segmento"] = data["SensorId"].map(sensor_map)
        data = rename_variables(data)

        for col in ["GyroX", "GyroY", "GyroZ","LinAccX", "LinAccY", "LinAccZ"]:
            if col in data.columns:
                data[col] = pd.to_numeric(data[col], errors="coerce")
                data[col] = data[col].interpolate()   # rellena huecos
                data[col] = filtro_pasabaja(data[col].values)

        data = calculate_mag(data)

        features = {}

        for segmento in ["Mano", "Bicep", "Antebrazo"]:

            seg = data[data["Segmento"] == segmento]

            if not seg.empty:
                features[f"LinAccMag_max_{segmento}"] = seg["LinAccMag"].max()
                features[f"LinAccMag_std_{segmento}"] = seg["LinAccMag"].std()
                features[f"GyroMag_max_{segmento}"] = seg["GyroMag"].max()
                features[f"GyroMag_std_{segmento}"] = seg["GyroMag"].std()

                idx_peak = seg["GyroMag"].idxmax()
                features[f"Time_to_peak_{segmento}"] = seg.loc[idx_peak, "TimeStamp"]

                features[f"TotalRotation_{segmento}"] = total_rotation(seg)
                features[f"RotEnergy_{segmento}"] = rotational_energy(seg)

            else:
                features[f"LinAccMag_max_{segmento}"] = np.nan
                features[f"LinAccMag_std_{segmento}"] = np.nan
                features[f"GyroMag_max_{segmento}"] = np.nan
                features[f"GyroMag_std_{segmento}"] = np.nan
                features[f"Time_to_peak_{segmento}"] = np.nan
                features[f"TotalRotation_{segmento}"] = np.nan
                features[f"RotEnergy_{segmento}"] = np.nan

        seg_mano = data[data["Segmento"] == "Mano"]
        seg_antebrazo = data[data["Segmento"] == "Antebrazo"]

        if not seg_mano.empty and not seg_antebrazo.empty:

            features["MaxRelRot_Mano_Antebrazo"] = max_relative_rotation(seg_mano, seg_antebrazo)

            if "Time_to_peak_Antebrazo" in features:
                t_release = features["Time_to_peak_Antebrazo"]
                features["Stability_Mano"] = stability_last300ms(seg_mano, t_release)
                features["TimePeakRot_Antebrazo"] = time_to_peak_rotation(seg_antebrazo)
            else:
                features["Stability_Mano"] = np.nan
                features["TimePeakRot_Antebrazo"] = np.nan

        else:
            features["MaxRelRot_Mano_Antebrazo"] = np.nan
            features["Stability_Mano"] = np.nan
            features["TimePeakRot_Antebrazo"] = np.nan


        features["Nombre Archivo"] = os.path.basename(archivo)

        all_features.append(features)

    return pd.DataFrame(all_features)



In [103]:
df_features = extract_features(csv_train)


Procesando: /home/josejuarez/Documents/Dataset_Inercial/train/Javier_05.csv


/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)



Procesando: /home/josejuarez/Documents/Dataset_Inercial/train/Erick_12.csv


/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)



Procesando: /home/josejuarez/Documents/Dataset_Inercial/train/Mariel_12.csv


...

/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)



Procesando: /home/josejuarez/Documents/Dataset_Inercial/train/Edgar_15.csv

Procesando: /home/josejuarez/Documents/Dataset_Inercial/train/Juliet_01.csv


/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)



Procesando: /home/josejuarez/Documents/Dataset_Inercial/train/Zarif_07.csv


/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)


In [104]:
df_features.to_csv("df_features.csv", index=False)
df_features.head()

,LinAccMag_max_Mano,LinAccMag_std_Mano,GyroMag_max_Mano,GyroMag_std_Mano,Time_to_peak_Mano,TotalRotation_Mano,RotEnergy_Mano,LinAccMag_max_Bicep,LinAccMag_std_Bicep,GyroMag_max_Bicep,...,LinAccMag_std_Antebrazo,GyroMag_max_Antebrazo,GyroMag_std_Antebrazo,Time_to_peak_Antebrazo,TotalRotation_Antebrazo,RotEnergy_Antebrazo,MaxRelRot_Mano_Antebrazo,Stability_Mano,TimePeakRot_Antebrazo,Nombre Archivo
0,4.703612,0.932354,997.142452,177.292863,0.6100,1.044429,5.356436,4.702263,1.087005,811.850727,...,0.716056,526.801158,118.697573,0.4301,0.789027,3.423881,2.194862,0.595569,0.8101,Javier_05.csv
1,6.219496,1.234962,867.388719,150.842259,0.8901,1.362660,3.193259,NaN,NaN,NaN,...,1.347840,869.974236,146.001977,0.9500,0.632630,1.008671,2.088434,0.526680,0.8000,Erick_12.csv
2,3.935244,0.639487,1073.051877,166.314397,0.8200,1.137026,5.810245,3.916724,0.729056,837.503902,...,0.695265,517.476350,112.875854,1.1300,1.003185,2.580353,2.618439,0.558406,0.8900,Mariel_12.csv
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.135834,683.929966,140.233493,0.5802,1.336719,3.311553,NaN,NaN,NaN,Zarif_08.csv
4,4.647633,0.794906,733.964185,136.570127,0.7400,1.782927,5.129809,4.876084,0.656520,747.539765,...,0.915232,680.147721,131.599928,0.7500,1.391554,2.487862,2.362627,0.569592,0.7000,Mariel_04.csv


In [105]:
df_labels = pd.read_csv("/home/josejuarez/Documents/Dataset_Inercial/labels.csv")
df_labels.head()

,Sujeto,clase_Tiro,Intento,Nombre Archivo,Unnamed: 4
0,Mariel,0,1,Mariel_01.csv,NaN
1,Mariel,0,2,Mariel_02.csv,NaN
2,Mariel,1,3,Mariel_03.csv,NaN
3,Mariel,0,4,Mariel_04.csv,NaN
4,Mariel,0,5,Mariel_05.csv,NaN


In [106]:
df_trainset = df_features.merge(df_labels, on="Nombre Archivo", how="left")


In [107]:
df_trainset.drop(columns=["Nombre Archivo", "Sujeto", "Intento","Unnamed: 4"],inplace=True)


In [108]:
df_trainset.head()

,LinAccMag_max_Mano,LinAccMag_std_Mano,GyroMag_max_Mano,GyroMag_std_Mano,Time_to_peak_Mano,TotalRotation_Mano,RotEnergy_Mano,LinAccMag_max_Bicep,LinAccMag_std_Bicep,GyroMag_max_Bicep,...,LinAccMag_std_Antebrazo,GyroMag_max_Antebrazo,GyroMag_std_Antebrazo,Time_to_peak_Antebrazo,TotalRotation_Antebrazo,RotEnergy_Antebrazo,MaxRelRot_Mano_Antebrazo,Stability_Mano,TimePeakRot_Antebrazo,clase_Tiro
0,4.703612,0.932354,997.142452,177.292863,0.6100,1.044429,5.356436,4.702263,1.087005,811.850727,...,0.716056,526.801158,118.697573,0.4301,0.789027,3.423881,2.194862,0.595569,0.8101,0
1,6.219496,1.234962,867.388719,150.842259,0.8901,1.362660,3.193259,NaN,NaN,NaN,...,1.347840,869.974236,146.001977,0.9500,0.632630,1.008671,2.088434,0.526680,0.8000,0
2,3.935244,0.639487,1073.051877,166.314397,0.8200,1.137026,5.810245,3.916724,0.729056,837.503902,...,0.695265,517.476350,112.875854,1.1300,1.003185,2.580353,2.618439,0.558406,0.8900,0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.135834,683.929966,140.233493,0.5802,1.336719,3.311553,NaN,NaN,NaN,0
4,4.647633,0.794906,733.964185,136.570127,0.7400,1.782927,5.129809,4.876084,0.656520,747.539765,...,0.915232,680.147721,131.599928,0.7500,1.391554,2.487862,2.362627,0.569592,0.7000,0


Una vez analizado y reunido el dataset completo, se tomo la decision de eliminar el conjunto que conformaba los datos del sensor colocado en el bicep. Esto debido a que presentaba una perdida de 78% de los datos,mostrandose en este caso como "NaN" y sin ser posible de alguna manera de reconstruirlo sin inventar datos o hacer que los modelos estuvieran mas sesgados a aprender cuando hay una medicion del bicep o cuando no hay.

In [109]:
cols_bicep = [col for col in df_trainset.columns if "Bicep" in col]
df_trainset = df_trainset.drop(columns=cols_bicep)


In [110]:
df_trainset.to_csv("df_trainset.csv")
df_trainset.head()

,LinAccMag_max_Mano,LinAccMag_std_Mano,GyroMag_max_Mano,GyroMag_std_Mano,Time_to_peak_Mano,TotalRotation_Mano,RotEnergy_Mano,LinAccMag_max_Antebrazo,LinAccMag_std_Antebrazo,GyroMag_max_Antebrazo,GyroMag_std_Antebrazo,Time_to_peak_Antebrazo,TotalRotation_Antebrazo,RotEnergy_Antebrazo,MaxRelRot_Mano_Antebrazo,Stability_Mano,TimePeakRot_Antebrazo,clase_Tiro
0,4.703612,0.932354,997.142452,177.292863,0.6100,1.044429,5.356436,4.739240,0.716056,526.801158,118.697573,0.4301,0.789027,3.423881,2.194862,0.595569,0.8101,0
1,6.219496,1.234962,867.388719,150.842259,0.8901,1.362660,3.193259,6.115632,1.347840,869.974236,146.001977,0.9500,0.632630,1.008671,2.088434,0.526680,0.8000,0
2,3.935244,0.639487,1073.051877,166.314397,0.8200,1.137026,5.810245,4.292603,0.695265,517.476350,112.875854,1.1300,1.003185,2.580353,2.618439,0.558406,0.8900,0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.826018,1.135834,683.929966,140.233493,0.5802,1.336719,3.311553,NaN,NaN,NaN,0
4,4.647633,0.794906,733.964185,136.570127,0.7400,1.782927,5.129809,4.978960,0.915232,680.147721,131.599928,0.7500,1.391554,2.487862,2.362627,0.569592,0.7000,0


Otra decision tomada durante la practica fue el hecho de modificar las clases de tiros, en este caso volver las clases 2 (Entro limpio) a las clases 1 (Entro), esto ya que al solo tener 4 muestras de estas, iba a hacer que fuera mas probable el que fueran catalogadas como de clase 0 (No Entro).

In [111]:
df_trainset["clase_Tiro"] = df_trainset["clase_Tiro"].replace({2:1})
df_trainset.to_csv("df_trainset.csv")


Por ultimo, se realizo una imputacion utilizando KNN, esto para eliminar los NaN que contenia el dataset, buscando los tiros similares entre todo el dataset y elige el mas parecido al que se busca rellenar, en este caso, solo buscamos los 5 tiros mas similares del que se detecta por medio del imputer

In [112]:
imputer = KNNImputer(n_neighbors=5)
df_trainset = pd.DataFrame(imputer.fit_transform(df_trainset), columns=df_trainset.columns)


In [113]:
df_trainset.to_csv("df_trainset.csv")
df_trainset.head()

,LinAccMag_max_Mano,LinAccMag_std_Mano,GyroMag_max_Mano,GyroMag_std_Mano,Time_to_peak_Mano,TotalRotation_Mano,RotEnergy_Mano,LinAccMag_max_Antebrazo,LinAccMag_std_Antebrazo,GyroMag_max_Antebrazo,GyroMag_std_Antebrazo,Time_to_peak_Antebrazo,TotalRotation_Antebrazo,RotEnergy_Antebrazo,MaxRelRot_Mano_Antebrazo,Stability_Mano,TimePeakRot_Antebrazo,clase_Tiro
0,4.703612,0.932354,997.142452,177.292863,0.6100,1.044429,5.356436,4.739240,0.716056,526.801158,118.697573,0.4301,0.789027,3.423881,2.194862,0.595569,0.81010,0.0
1,6.219496,1.234962,867.388719,150.842259,0.8901,1.362660,3.193259,6.115632,1.347840,869.974236,146.001977,0.9500,0.632630,1.008671,2.088434,0.526680,0.80000,0.0
2,3.935244,0.639487,1073.051877,166.314397,0.8200,1.137026,5.810245,4.292603,0.695265,517.476350,112.875854,1.1300,1.003185,2.580353,2.618439,0.558406,0.89000,0.0
3,8.884093,1.881717,1112.258627,277.773422,0.8900,1.194239,4.074051,4.826018,1.135834,683.929966,140.233493,0.5802,1.336719,3.311553,2.358798,0.532317,0.83802,0.0
4,4.647633,0.794906,733.964185,136.570127,0.7400,1.782927,5.129809,4.978960,0.915232,680.147721,131.599928,0.7500,1.391554,2.487862,2.362627,0.569592,0.70000,0.0


<h2 style="color: #5439a7ff; text-align: center;">
  Evaluacion interna
</h3>

In [116]:
X = df_trainset.drop(columns=["clase_Tiro"])
y = df_trainset["clase_Tiro"]

In [117]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [119]:
cv = StratifiedKFold(
    n_splits=5,      
    shuffle=True,
    random_state=42
)

In [208]:
models = {
    "Logistic": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=300,max_depth=10,min_samples_split=5,random_state=42, class_weight="balanced"),
    "SVM": SVC(C=10,gamma="scale",kernel="rbf",class_weight="balanced"),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=42, class_weight="balanced")
}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
    print(f"{name}: {scores.mean():.4f}")

/home/josejuarez/cdsi_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/josejuarez/cdsi_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/prepr

Logistic: 0.4580
RandomForest: 0.5461
SVM: 0.4274
KNN: 0.4939
DecisionTree: 0.5960


In [209]:
best_model = DecisionTreeClassifier()
best_model.fit(X_scaled, y)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [210]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = best_model.predict(X_scaled)

print(confusion_matrix(y, y_pred))
print(classification_report(y, y_pred))


[[88  0]
 [ 0 23]]
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00        88
         1.0       1.00      1.00      1.00        23

    accuracy                           1.00       111
   macro avg       1.00      1.00      1.00       111
weighted avg       1.00      1.00      1.00       111



<h2 style="color: #5439a7ff; text-align: center;">
    Validacion externa
</h3>

In [211]:
df_testset = extract_features(csv_test)


Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Tony_04.csv


/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)



Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Joan_13.csv

Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Tony_01.csv


/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)



Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Joan_07.csv

Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Joan_20.csv

Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Joan_04.csv

Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Joan_18.csv


...

/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)



Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Joan_03.csv


/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)



Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Joan_15.csv

Procesando: /home/josejuarez/Documents/Dataset_Inercial/test/Joan_02.csv


/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)
/tmp/ipykernel_30279/3058320297.py:21: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(angles**2, times)


In [212]:
df_testset.to_csv("df_testset.csv", index=False)
df_testset.head()

,LinAccMag_max_Mano,LinAccMag_std_Mano,GyroMag_max_Mano,GyroMag_std_Mano,Time_to_peak_Mano,TotalRotation_Mano,RotEnergy_Mano,LinAccMag_max_Bicep,LinAccMag_std_Bicep,GyroMag_max_Bicep,...,LinAccMag_std_Antebrazo,GyroMag_max_Antebrazo,GyroMag_std_Antebrazo,Time_to_peak_Antebrazo,TotalRotation_Antebrazo,RotEnergy_Antebrazo,MaxRelRot_Mano_Antebrazo,Stability_Mano,TimePeakRot_Antebrazo,Nombre Archivo
0,NaN,NaN,998.730213,211.309938,0.6701,1.322826,4.406872,NaN,NaN,NaN,...,NaN,733.698065,152.081911,0.3495,0.791887,3.850248,3.133687,0.745660,1.0195,Tony_04.csv
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.514896,1.291502,775.759179,...,0.953827,739.881346,129.627059,0.2899,1.812316,2.420510,NaN,NaN,NaN,Joan_13.csv
2,NaN,NaN,504.945932,144.413228,0.6701,1.110196,2.676636,NaN,NaN,NaN,...,NaN,499.323356,104.403932,1.1296,1.964685,3.360822,3.133439,0.033708,1.4496,Tony_01.csv
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.227435,590.466282,151.852873,0.6698,2.108836,3.081930,NaN,NaN,NaN,Joan_07.csv
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.231575,304.635966,94.765915,0.5999,2.016822,2.781758,NaN,NaN,NaN,Joan_20.csv


In [213]:
cols_bicep = [col for col in df_testset.columns if "Bicep" in col]
df_testset = df_testset.drop(columns=cols_bicep)

In [214]:
df_testset.head()
df_testset.to_csv("df_testset.csv")

In [215]:
X_test = df_testset.drop(columns=["Nombre Archivo"])

In [216]:
X_test = pd.DataFrame(imputer.fit_transform(X_test),columns=X_test.columns)
X_test = scaler.transform(X_test)

In [217]:
y_pred = best_model.predict(X_test)


In [218]:
df_testset["Prediccion"] = y_pred
df_testset[["Nombre Archivo", "Prediccion"]]


,Nombre Archivo,Prediccion
0,Tony_04.csv,0.0
1,Joan_13.csv,0.0
2,Tony_01.csv,1.0
3,Joan_07.csv,0.0
4,Joan_20.csv,0.0
5,Joan_04.csv,0.0
6,Joan_18.csv,0.0
7,Tony_07.csv,0.0
8,Joan_06.csv,0.0
9,Joan_14.csv,0.0


<h2 style="color: #5439a7ff; text-align: center;">
  Conclusiones
</h3>